In [3]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS

load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

print("환경설정 완료")


/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


환경설정 완료


In [4]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]


In [5]:
!pip uninstall numpy pandas -y
!pip install numpy==1.24.4 pandas==1.5.3

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 3.0.2
Uninstalling pandas-3.0.2:
  Successfully uninstalled pandas-3.0.2
  Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
  Using cached pandas-1.5.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
Using cached pandas-1.5.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires numpy>=1.26.2; python_version < "3.13", but you have numpy 1.24.4 which is incompatible.
faiss-cpu 1.13.2 requires numpy<3.0,>=1.25.0, but you have numpy 1.24.4 which is inc

In [8]:
pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [9]:
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [11]:
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [ ]:
pip install langchain openai

In [ ]:
# ranked  -> (Cross-encoder / LLM) -> reranked
# vector search: 쿼리 / 문서: Bi encode
# BM25 -> rerank
# (쿼리 - 문서) -> rerank
# LLM -> rerank

In [12]:
import numpy as np
import math
from collections import Counter

class BM25Reranker:
    def __init__(self, k1=1.5, b=0.75):
        # 파이썬은 대소문자를 구분하므로 생성자 인자와 멤버 변수 이름을 맞춥니다.
        self.k1 = k1
        self.b = b

    def rerank(self, query, search_results):
        if not search_results:
            return []

        # 1. 문서 추출 및 토큰화
        docs = [doc for doc, _ in search_results]
        tokenized = [doc.page_content.lower().split() for doc in docs]
        
        # 2. 통계치 계산 (N, 평균 문서 길이, DF)
        N = len(docs)
        # avg_dl (Average Document Length) 계산 - 오타 수정 (let -> len)
        avg_dl = np.mean([len(t) for t in tokenized])

        df_count = Counter()
        for tokens in tokenized:
            # 한 문서 내 중복 제거 후 세어서 Document Frequency 계산
            for t in set(tokens):
                df_count[t] += 1

        # 3. 쿼리 토큰화
        query_tokens = query.lower().split()
        scored = []

        # 4. 각 문서별 스코어링
        for i, doc in enumerate(docs):
            score = 0.0
            current_tokens = tokenized[i]
            doc_len = len(current_tokens)
            tf_counter = Counter(current_tokens)

            for qt in query_tokens:
                tf = tf_counter.get(qt, 0) # 변수명 오타 수정 (tf_count -> tf_counter)
                if tf == 0:
                    continue

                df = df_count.get(qt, 0)
                
                # BM25용 IDF 공식
                idf = math.log((N - df + 0.5) / (df + 0.5) + 1.0)
                
                # BM25 TF 정규화 공식 적용
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * (doc_len / avg_dl))
                
                score += idf * (numerator / denominator)

            scored.append((doc, score))

        # 5. 점수 기준 내림차순 정렬 (x[1] 기준)
        scored.sort(key=lambda x: x[1], reverse=True)

        return scored

In [14]:
def llm_rerank(query, search_results, llm, top_k=3):
    """
    LLM을 사용하여 검색 결과의 관련성을 재평가(Rerank)합니다.
    """
    if not search_results:
        return []

    # 1. 문서 텍스트 및 메타데이터 준비 (f-string 따옴표 충돌 방지를 위해 외부에 정의)
    docs_text = "\n".join(
        f"[{doc.metadata.get('id', 'N/A')}] {doc.page_content}" for doc, _ in search_results
    )

    # 2. JSON 스코어 템플릿 생성
    score_template = ", ".join(f'"{doc.metadata.get("id", "N/A")}": 0.0-1.0' for doc, _ in search_results)

    # 3. 프롬프트 구성 및 체인 생성
    # ChatPromptTemplate 구성 시 쉼표(,) 누락 수정 및 구조 개선
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 관련성 평가 전문가입니다. 반드시 제공된 JSON 형식으로만 답변하세요."),
        ("human", """다음 쿼리에 대해 각 문서의 관련성을 0.0에서 1.0 사이로 평가하세요. 1.0은 매우 관련됨, 0.0은 관련 없음을 의미합니다.

쿼리: {query}

문서 리스트:
{docs_text}

JSON 응답 예시:
{{
    "scores": {{
        {score_template}
    }}
}}
""")
    ])

    # JsonOutputParser를 사용하여 안정성 확보
    scoring_chain = prompt | llm | JsonOutputParser()

    try:
        # 4. LLM 호출 및 결과 파싱
        parsed = scoring_chain.invoke({
            "query": query,
            "docs_text": docs_text,
            "score_template": score_template
        })

        scores = parsed.get("scores", {})
        scored = []

        # 5. 원래 문서 객체와 새로운 점수 매칭
        for doc, _ in search_results:
            doc_id = str(doc.metadata.get("id", ""))
            # LLM이 문자열 키로 반환할 수 있으므로 문자열로 매칭
            rerank_score = float(scores.get(doc_id, 0.0))
            scored.append((doc, rerank_score))

        # 6. 점수 기준 정렬 및 top_k 추출
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    except Exception as e:
        print(f"LLM Rerank 에러 발생: {e}")
        return search_results[:top_k] # 에러 발생 시 원래 순서대로 반환

In [ ]:
import math
from collections import Counter

class BM25Reranker:
    def __init__(self, k1=1.5, b=0.75):
        # 파이썬은 대소문자를 구분하므로 생성자 인자와 멤버 변수 이름을 맞춥니다.
        self.k1 = k1
        self.b = b

    def rerank(self, query, search_results):
        if not search_results:
            return []

        # 1. 문서 추출 및 토큰화
        docs = [doc for doc, _ in search_results]
        tokenized = [doc.page_content.lower().split() for doc in docs]
        
        # 2. 통계치 계산 (N, 평균 문서 길이, DF)
        N = len(docs)
        # avg_dl (Average Document Length) 계산 - 오타 수정 (let -> len)
        avg_dl = np.mean([len(t) for t in tokenized])

        df_count = Counter()
        for tokens in tokenized:
            # 한 문서 내 중복 제거 후 세어서 Document Frequency 계산
            for t in set(tokens):
                df_count[t] += 1

        # 3. 쿼리 토큰화
        query_tokens = query.lower().split()
        scored = []

        # 4. 각 문서별 스코어링
        for i, doc in enumerate(docs):
            score = 0.0
            current_tokens = tokenized[i]
            doc_len = len(current_tokens)
            tf_counter = Counter(current_tokens)

            for qt in query_tokens:
                tf = tf_counter.get(qt, 0) # 변수명 오타 수정 (tf_count -> tf_counter)
                if tf == 0:
                    continue

                df = df_count.get(qt, 0)
                
                # BM25용 IDF 공식
                idf = math.log((N - df + 0.5) / (df + 0.5) + 1.0)
                
                # BM25 TF 정규화 공식 적용
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * (doc_len / avg_dl))
                
                score += idf * (numerator / denominator)

            scored.append((doc, score))

        # 5. 점수 기준 내림차순 정렬 (x[1] 기준)
        scored.sort(key=lambda x: x[1], reverse=True)

        return scored

class ScoreFilter:
    
    def fixed_threshold(scored_docs, threshold=0.5):
        """고정 임계값 이상의 문서만 반환"""
        return [(doc, s) for doc, s in scored_docs if s >= threshold]

    
    def dynamic_threshold(scored_docs, std_factor=1.0):
        """평균과 표준편차를 이용한 동적 임계값 적용"""
        if not scored_docs:
            return []
        scores = [s for _, s in scored_docs]
        threshold = np.mean(scores) - std_factor * np.std(scores)
        return [(doc, s) for doc, s in scored_docs if s >= threshold]

    
    def score_gap(scored_docs, min_docs=2):
        """가장 큰 점수 차이가 발생하는 지점에서 컷오프"""
        if len(scored_docs) <= min_docs:
            return scored_docs

        scores = [s for _, s in scored_docs]
        # 점수 차이(gap)와 다음 인덱스를 저장
        gaps = [(scores[i] - scores[i+1], i+1) for i in range(len(scores) - 1)]
        
        # 가장 큰 gap을 가진 인덱스 찾기
        max_gap_idx = max(gaps, key=lambda x: x[0])[1]
        
        # 최소 문서 수(min_docs)와 gap 지점 중 큰 값을 선택하여 컷
        cut = max(min_docs, max_gap_idx)
        return scored_docs[:cut]

In [16]:
# 1. 리랭커 객체 생성
reranker = BM25Reranker()

query = "트랜스포머는 무엇인가요?"
search_results = [
    (MocDoc("트랜스포머는 생성형 AI 엔진입")
]


# 2. 리랭킹 실행 (이 결과값이 scored_docs가 됩니다)
# query와 search_results는 사전에 정의되어 있어야 합니다.
scored_docs = reranker.rerank(query, search_results)

# 3. 임계값 변수 정의 (오타 주의: threshold)
threshold_value = 0.5

sf = ScoreFilter()
sf.fixed_threshold(scored_docs, thresold)

NameError: name 'search_results' is not defined

In [ ]:
test_scores = [(Document(page_content="", metadata= {'id': f'd{i}'}), s) for i, s in enumerate([0.9, 0.85, 0.7, 0.5, 0.2])]
test_scores

In [ ]:
filtered = ScoreFilter.fixed_thresold(test_scores, thresold=0.5)
filtered

In [ ]:
filtered = ScoreFilter.fixed_thresold(score_gap, )

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Optional

@dataclass
class ScoredDoc:
    doc: str
    score: float

class AdaptiveFilter:
    """점수의 분포에 따라 score filtering 전략을 자동으로 선택"""

    def __init__(self, score_gap_threshold: float = 0.15, std_factor: float = 1.0):
        self.score_gap_threshold = score_gap_threshold  # score_gap 전략에서 gap 기준
        self.std_factor = std_factor                    # dynamic_threshold 기본 factor
        
    @staticmethod
    def filter(self, scored_docs: list[ScoredDoc]) -> list[ScoredDoc]:
        if not scored_docs:
            return []

        scores = np.array([d.score for d in scored_docs])
        std = float(np.std(scores))
        score_range = float(np.max(scores) - np.min(scores))

        # 전략 선택
        if std > 0.2:
            strategy = "score_gap"
        elif score_range < 0.1:
            strategy = "top_half"
        else:
            strategy = "dynamic_threshold"

        result = self._apply_strategy(scored_docs, scores, std, strategy)

        print(f"[AdaptiveFilter] std={std:.3f}, range={score_range:.3f} → strategy: {strategy} ({len(result)}/{len(scored_docs)} docs passed)")
        return result

    def _apply_strategy(
        self,
        scored_docs: list[ScoredDoc],
        scores: np.ndarray,
        std: float,
        strategy: str,
    ) -> list[ScoredDoc]:

        if strategy == "score_gap":
            return self._score_gap(scored_docs, scores)

        elif strategy == "top_half":
            return self._top_half(scored_docs)

        else:  # dynamic_threshold
            return self._dynamic_threshold(scored_docs, scores, std, self.std_factor)

    # ── 전략 구현 ────────────────────────────────────────────

    def _score_gap(self, scored_docs: list[ScoredDoc], scores: np.ndarray) -> list[ScoredDoc]:
        """점수를 내림차순 정렬 후 가장 큰 gap이 발생하는 지점에서 분리"""
        sorted_docs = sorted(scored_docs, key=lambda d: d.score, reverse=True)
        sorted_scores = np.sort(scores)[::-1]

        gaps = np.diff(sorted_scores)           # 인접 점수 간 차이 (음수)
        gap_idx = int(np.argmin(gaps))          # gap이 가장 큰 인덱스

        # gap이 threshold보다 작으면 전체 반환 (의미 있는 gap 없음)
        if abs(gaps[gap_idx]) < self.score_gap_threshold:
            return sorted_docs

        return sorted_docs[: gap_idx + 1]

    def _top_half(self, scored_docs: list[ScoredDoc]) -> list[ScoredDoc]:
        """score_range가 좁을 때 → 단순 상위 50% 반환"""
        sorted_docs = sorted(scored_docs, key=lambda d: d.score, reverse=True)
        cutoff = max(1, len(sorted_docs) // 2)
        
        return sorted_docs[:cutoff]

    def _dynamic_threshold(
        self,
        scored_docs: list[ScoredDoc],
        scores: np.ndarray,
        std: float,
        std_factor: float,
    ) -> list[ScoredDoc]:
        """mean - std_factor * std 이상인 문서만 통과"""
        threshold = float(np.mean(scores) - std_factor * std)
        
        return [d for d in scored_docs if d.score >= threshold]

In [ ]:
filtered = AdaptiveFilter.filter(test_scores)
filtered

In [ ]:
# 컨텍스트 메모리가 80% 찼을때 압축해라. 
# 턴이 20턴 정도 끝나면 압축해라. 
# 잘하고 내가 편하다 -> 위험해진다. 주권이 넘어간다. 영향력이 커진다. 규칙 거버넌스, 규제가이드라인을 빡빡하게 준다. 
# 최소 권한 워칙, 결제 트랜잭션 최소화, 개인정보 최소화
# 정보를 감출수록 -> 불편해진다. 

In [ ]:
#mmr: retreiver : Embedding, BM25, HybridRetriever

In [ ]:
## Relevance <----> Diversity 다양성, 일반화. mmr을 높이면 다양하게 가져온다?

In [ ]:
P = 0.1, 0.2,....
mmr = P * Relevance(s, query) - (1-P)*max(Similarity(s, selected)

In [ ]:
def extractive_compress(query, document, max_sentence=3):
    sentences = re.split(r'[.!?\s', document)

    if not sentences:
        return document

    query_terms = set(query.lower().split())

    scored = []

    for sent in sentences:
        sent_terms = set(sent.lower().split())
        overlap = len(query_terms & sent_terms)
        score.appen((sent, overlap))

    scored.sort(key=lambda x:x[1], reverse=True)
    selected = [s for s, _ in scored[:max_sentences]]

    return ", ".join(selected) + "."

In [ ]:
long_doc = "트랜스포머는 2017년 구글이 발표한 아키텍쳐입니다. "
query = "트랜스포머와 BERT의 관계"
compressed = exractive_compress(query, long_doc, max_sentences=3)

In [ ]:
compress_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서 요약 전문가 입니다."),
    ("human", """ 다음 문서에서 쿼리에 답하는데 필요한 핵심 정보만 추출하세요.
    불필요한 내용은 제거하고, {max_tokens}자 이내로 압축하세요.

    쿼리: {query}
    문서: {document}

    압축결과:
    """)
]) | llm | StrOutParser()

In [ ]:
def llm_compress(query, document, max_tokens=100):
    return compress_chain.invoke({
        'query': query,
        'document': document,
        'max_tokens': max_tokens
    })

In [ ]:
compressed_file = llm_compress(query, lang_doc)
compressed_llm 

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)
compressor_retriever = ContextCompressionRetriver(base_compressor=compressor, base_retriever=vectorstore.as_retriever()
search_kwargs={'k': 3})

In [ ]:
compressed_docs = compressor_retriever.invoke(query)

In [ ]:
@dataclass
class CompressionEvalResult:
    compression_ratio: float        # 낮을수록 압축 많이 됨
    keyword_coverage: float         # 1.0 = 쿼리 키워드 100% 보존
    semantic_similarity: float      # 1.0 = 의미 완전 보존
    quality_score: float            # 종합 점수
    char_original: int
    char_compressed: int
    warnings: list[str] = field(default_factory=list)

    def __str__(self) -> str:
        lines = [
            f"  압축률           : {self.compression_ratio:.2%} ({self.char_original}자 → {self.char_compressed}자)",
            f"  키워드 보존율    : {self.keyword_coverage:.2%}",
            f"  의미 보존도      : {self.semantic_similarity:.4f}",
            f"  종합 품질 점수   : {self.quality_score:.4f}",
        ]
        if self.warnings:
            lines += [f"  ⚠️  {w}" for w in self.warnings]
        return "\n".join(lines)


def _tokenize(text: str) -> set[str]:
    """한국어/영어 혼용 토크나이징 (구두점 제거)"""
    return set(re.findall(r"[가-힣]+|[a-z0-9]+", text.lower()))


def _compression_ratio(original: str, compressed: str) -> float:
    """압축 후 길이 / 원본 길이 (낮을수록 더 압축)"""
    if len(original) == 0:
        return 1.0
    return len(compressed) / len(original)


def _keyword_coverage(original: str, compressed: str, query: str) -> float:
    """쿼리 키워드 중 원본에 있던 것들이 압축본에 얼마나 남았는지"""
    query_terms = _tokenize(query)
    orig_terms  = _tokenize(original)
    comp_terms  = _tokenize(compressed)

    orig_query_terms = query_terms & orig_terms   # 원본에 실제 존재했던 쿼리 토큰
    if not orig_query_terms:
        return 1.0  # 원본에 쿼리 키워드 자체가 없으면 패널티 없음

    comp_query_terms = orig_query_terms & comp_terms
    return len(comp_query_terms) / len(orig_query_terms)


def _semantic_similarity(original: str, compressed: str) -> float:
    """Sentence Embedding 기반 코사인 유사도"""
    model = get_model()
    embs = model.encode([original, compressed], convert_to_numpy=True)
    sim = cosine_similarity(embs[0:1], embs[1:2])
    return float(sim[0][0])


def _quality_score(
    ratio: float,
    keyword_coverage: float,
    semantic_similarity: float,
    w_compression: float = 0.2,
    w_keyword: float = 0.3,
    w_semantic: float = 0.5,
) -> float:
    """
    종합 품질 점수 (0~1, 높을수록 좋음)
    - compression_ratio는 낮을수록 좋으므로 (1 - ratio)로 반전
    - 가중치 합산
    """
    compression_score = 1.0 - ratio   # 많이 압축될수록 높은 점수
    return (
        w_compression * compression_score
        + w_keyword   * keyword_coverage
        + w_semantic  * semantic_similarity
    )


def _collect_warnings(
    ratio: float,
    keyword_coverage: float,
    semantic_similarity: float,
) -> list[str]:
    warnings = []
    if ratio > 0.9:
        warnings.append("압축이 거의 이루어지지 않았습니다 (ratio > 0.9)")
    if ratio < 0.1:
        warnings.append("과도한 압축 — 정보 손실 위험 (ratio < 0.1)")
    if keyword_coverage < 0.5:
        warnings.append("쿼리 핵심 키워드 절반 이상 소실")
    if semantic_similarity < 0.7:
        warnings.append("의미 보존도 낮음 — 압축 방식 재검토 권장")
    return warnings


# ── 메인 함수 ────────────────────────────────────────────────

def evaluate_compression(
    original: str,
    compressed: str,
    query: str,
    w_compression: float = 0.2,
    w_keyword: float = 0.3,
    w_semantic: float = 0.5,
    verbose: bool = False,
) -> CompressionEvalResult:
    """
    압축 품질 종합 평가

    Args:
        original    : 원본 텍스트
        compressed  : 압축된 텍스트
        query       : 검색 쿼리 (키워드 보존율 기준)
        w_*         : 종합 점수 가중치 (합 = 1.0)
        verbose     : True면 결과를 콘솔 출력
    """
    if not original.strip():
        raise ValueError("original 텍스트가 비어 있습니다.")
    if not compressed.strip():
        raise ValueError("compressed 텍스트가 비어 있습니다.")

    ratio     = _compression_ratio(original, compressed)
    kw_cov    = _keyword_coverage(original, compressed, query)
    sem_sim   = _semantic_similarity(original, compressed)
    q_score   = _quality_score(ratio, kw_cov, sem_sim, w_compression, w_keyword, w_semantic)
    warnings  = _collect_warnings(ratio, kw_cov, sem_sim)

    result = CompressionEvalResult(
        compression_ratio   = ratio,
        keyword_coverage    = kw_cov,
        semantic_similarity = sem_sim,
        quality_score       = q_score,
        char_original       = len(original),
        char_compressed     = len(compressed),
        warnings            = warnings,
    )

    if verbose:
        print("[CompressionEval]\n" + str(result))

    return result


In [ ]:
def mmr_compress(query, docuent, max_sentence=3, param=0.5):
    sentences = re.split(r'[.?!\s*', document)
    sentences = [s.strip() for s in sentences if len(s.strip > 10)]

    if len(sentences) <= max_sentences:
        return ".".join(sentences) + "."

    query_emb = get_embedding(query)
    sent_embs = [get_embedding(s) for s in sentences]

    selected = []
    remaining = list(range(sentences))

    for _ in range(max_sentences):
        best_score = -float('inf')

        for idx in remainig:
            relevance = cosine_sim(query_emb, sent_embs[idx])

            if selected:
                max_sim = max(cosine_sim(sent_embs[idx], sent_embs[s]) for s in selected)
            else:
                max_sim = 0.0

            mmr = param * relevance - (1 - param) * max_sim

            if mmr > best_score:
                best_score = mmr
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return ",".join(sentencesp[i] for i in sorted(selected)) + "."



In [ ]:
mmr_compress(query, long_doc, max_sentences=2, param=0.7)
mmr_compress(query, long_doc, max_sentences=2, param=0.3)

In [ ]:
리랭킹 효과
Precision - Average Precision - mAP 

BERT [관련 O, 관련 X, 관련 O, 관련 X, 관련 O]
Precision 1/1   0   2/3    0      3/5
AP        (1/1  +  2/3     +      3/5) / 3 = 0.75555

BERT [관련 O, 관련 X, 관련 O, 관련 X, 관련 O]
Precision  1/1  